# 6 — Déploiement (CRISP-DM Phase 6)

La Phase 5 a **désigné** le champion ; la Phase 6 le **met en production**. Ce notebook ne rejuge pas le modèle : il le reconstruit, le réentraîne sur 100 % des données, génère la **soumission Kaggle**, l'**enregistre dans MLflow** (registre + service en API), puis pose la **stratégie de déploiement**, le **coût** et un **PoC** (bonus). La synthèse exécutive et le récapitulatif complet sont dans le NB7 (conclusion).

## 6.1 Reconstruction et réentraînement du champion sur 100 % des données

On part du champion désigné par la Phase 5 (`results/winning_model.json`), on le reconstruit via un *dispatcher* (nom → Pipeline), puis on le **réentraîne sur `X` complet** (1460 biens, pas seulement les 80 % de `X_train`) pour la soumission. Le préprocesseur reste celui défini dans `2_data_prep.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

Dimensions brutes : (1460, 81)
Nombre de points supprimés : 2
Dimensions après suppression : (1458, 81)
Variables après ingénierie : 90 colonnes (+8 dérivées)
Corrélation des variables dérivées avec SalePrice_log :
  TotalSF            : +0.825
  TotalBathrooms     : +0.677
  HouseAge           : -0.588
  YearsSinceRemodel  : -0.569
  GarageAge          : -0.543
  HasGarage          : +0.323
  HasSecondFloor     : +0.151
  HasPool            : +0.077   <-- faible (|r| < 0.1)
X_train : (1166, 83)   |   X_test : (292, 83)
Colonnes retirées (colinéarité) : ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']
NaN dans X_train : 6276 cellules sur 96778
Audit de cardinalité des colonnes nominales (sur X_train) :
Colonne           modalités   bucket
  Neighborhood           25     TargetEncoder (haute)  [imput. mode]
  Exterior2nd            16     TargetEncoder (haute)  [imput. mode]
  Exterior1st            15     TargetEncoder (haute)  [imput. mode]
  MSSubClass             15     

In [2]:
winner_path = RESULTS_DIR / 'winning_model.json'
if not winner_path.exists():
    raise RuntimeError(
        "results/winning_model.json absent. Exécuter 4_evaluation.ipynb d'abord."
    )

winner = json.loads(winner_path.read_text())
print(f"Modèle champion désigné par 4_evaluation : {winner['model']}  (famille : {winner['family']})")
print(f"  Holdout RMSLE : {winner['holdout_rmsle']:.4f}")
print(f"  Params        : {winner.get('params')}")

Modèle champion désigné par 4_evaluation : Stacking  (famille : stacking)
  Holdout RMSLE : 0.1122
  Params        : {'meta': 'RidgeCV', 'base': ['Lasso', 'GBR', 'XGB']}


In [3]:
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
# CatBoostRegressorCV + CatBoostPrep proviennent de 2_data_prep (via %run).


def _scaled(model):
    return Pipeline([('preprocessor', preprocessor_scaled), ('model', model)])

def _encoded(model):
    return Pipeline([('preprocessor', preprocessor_encoded), ('model', model)])

def _native(model):
    return Pipeline([('preprocessor', preprocessor_native), ('model', model)])


def build_champion(model_name: str, params: dict | None):
    p = params or {}

    if model_name == 'OLS':
        return _scaled(LinearRegression())
    if model_name == 'Ridge':
        return _scaled(RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5))
    if model_name == 'Lasso':
        return _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'ElasticNet':
        return _scaled(ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                                    alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'KNN':
        return _scaled(KNeighborsRegressor(n_neighbors=int(p.get('k', 9))))
    if model_name == 'MLPRegressor':
        return _scaled(MLPRegressor(hidden_layer_sizes=(64,), activation='relu', solver='adam',
                                    max_iter=1500, early_stopping=True, validation_fraction=0.15,
                                    random_state=RANDOM_STATE))

    if model_name == 'DecisionTree':
        return _encoded(DecisionTreeRegressor(max_depth=int(p.get('max_depth', 6)), random_state=RANDOM_STATE))
    if model_name == 'RandomForest':
        return _encoded(RandomForestRegressor(n_estimators=int(p.get('n_estimators', 300)),
                                              random_state=RANDOM_STATE, n_jobs=-1))
    if model_name == 'GradientBoosting':
        return _encoded(GradientBoostingRegressor(
            n_estimators=int(p.get('n_estimators', 300)),
            learning_rate=float(p.get('learning_rate', 0.05)),
            max_depth=int(p.get('max_depth', 3)),
            random_state=RANDOM_STATE,
        ))
    if model_name == 'AdaBoost':
        return _encoded(AdaBoostRegressor(n_estimators=int(p.get('n_estimators', 200)), random_state=RANDOM_STATE))

    if model_name in {'XGBoost_native', 'XGBoost_onehot', 'XGBoost_tuned'}:
        defaults = dict(n_estimators=500, learning_rate=0.05, max_depth=4)
        cfg = {**defaults, **{k: v for k, v in p.items() if k in {
            'n_estimators', 'learning_rate', 'max_depth',
            'min_child_weight', 'subsample', 'colsample_bytree',
            'reg_alpha', 'reg_lambda',
        }}}
        # XGBoost_onehot used preprocessor_encoded; the others use preprocessor_native
        prep = preprocessor_encoded if model_name == 'XGBoost_onehot' else preprocessor_native
        kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, tree_method='hist')
        if prep is preprocessor_native:
            kwargs['enable_categorical'] = True
        return Pipeline([('preprocessor', prep), ('model', XGBRegressor(**cfg, **kwargs))])

    if model_name == 'LightGBM':
        return _native(LGBMRegressor(
            n_estimators=int(p.get('n_estimators', 500)),
            learning_rate=float(p.get('learning_rate', 0.05)),
            num_leaves=int(p.get('num_leaves', 31)),
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))

    if model_name == 'CatBoost':
        cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
        return Pipeline([
            ('preprocessor', CatBoostPrep()),
            ('model', CatBoostRegressorCV(
                cat_features=cat_cols,
                iterations=int(p.get('iterations', 500)),
                learning_rate=float(p.get('learning_rate', 0.05)),
                depth=int(p.get('depth', 6)),
                random_seed=RANDOM_STATE)),
        ])

    if model_name == 'Stacking':
        lasso_sub = _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
        gb_sub    = _encoded(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
        xgb_sub   = _native(XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                         enable_categorical=True, tree_method='hist',
                                         random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
        return StackingRegressor(
            estimators=[('lasso', lasso_sub), ('gbr', gb_sub), ('xgb', xgb_sub)],
            final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 10)),
            cv=5, n_jobs=-1, passthrough=False,
        )

    raise ValueError(f"Modèle inconnu dans le dispatcher : {model_name}")


champion = build_champion(winner['model'], winner.get('params'))
print(f"Champion reconstruit : {type(champion).__name__}")

Champion reconstruit : StackingRegressor


In [4]:
t0 = time.time()
champion.fit(X, y_log)
fit_s = time.time() - t0
print(f"Champion entraîné sur {len(X)} observations en {fit_s:.1f}s")

Champion entraîné sur 1458 observations en 13.4s


## 6.2 Soumission Kaggle

On charge `data/test.csv` (le **vrai** test set Kaggle, sans `SalePrice`), on applique exactement les mêmes drops de colonnes que sur le train (`Id`, `SalePrice*`), puis on prédit en espace log avant retransformation `expm1`.

In [5]:
df_test = pd.read_csv('./data/test.csv', sep=',')
test_ids = df_test['Id']
print(f"Test Kaggle : {df_test.shape}")

# Mêmes variables dérivées + cast MSSubClass que sur le train (fonction partagée du NB2).
# Le reindex sur X.columns applique aussi le retrait des colonnes colinéaires.
X_test_kaggle = engineer_features(df_test).drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
X_test_kaggle = X_test_kaggle[X.columns]

preds_log = champion.predict(X_test_kaggle)
preds_dollars = np.expm1(preds_log)

print(f"Statistiques des prédictions ($) :")
print(f"  min: {preds_dollars.min():,.0f}")
print(f"  mean: {preds_dollars.mean():,.0f}")
print(f"  max: {preds_dollars.max():,.0f}")
print(f"  nb NaN: {int(np.isnan(preds_dollars).sum())}")

assert not np.isnan(preds_dollars).any(), "Prédictions contiennent des NaN — modèle ou prétraitement défaillant"
assert (preds_dollars > 0).all(), "Prédictions <= 0 — anomalie"
assert len(preds_dollars) == len(df_test), "Longueur de prédictions ≠ longueur du test set"

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_dollars})
submission.to_csv('submission.csv', index=False)
print(f"\nSoumission écrite : submission.csv ({len(submission)} lignes)")
display(submission.head())

Test Kaggle : (1459, 80)
Statistiques des prédictions ($) :
  min: 46,443
  mean: 179,047
  max: 1,123,334
  nb NaN: 0

Soumission écrite : submission.csv (1459 lignes)


,Id,SalePrice
0,1461,117689.646470
1,1462,157865.292195
2,1463,180492.745364
3,1464,195888.388646
4,1465,190536.060290


## 6.3 MLflow — registre du modèle et service

Le champion réentraîné est enregistré dans le **MLflow Model Registry** sous le nom `inved-house-price`, avec une **signature** (schéma d'entrée/sortie inférée) et un *input example*. On promeut la version via l'alias **`@Production`** : c'est la référence canonique pour tout consommateur (API de service, PoC, jobs de réentraînement).

C'est la **moitié « déploiement » de MLflow** (registre + service) ; la moitié « traçabilité des expériences » (les 17 modèles candidats) est en Phase 7 (§7.2), à sa juste place — la surveillance.

In [6]:
import shutil
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

mlflow.autolog(disable=True)

MLRUNS = Path.cwd() / 'mlruns'
EXPERIMENT = 'inved-house-price'
REGISTERED_MODEL = 'inved-house-price'

# Idempotence : registre propre à chaque exécution complète du pipeline (NB5 avant NB6)
if MLRUNS.exists():
    shutil.rmtree(MLRUNS)
mlflow.set_tracking_uri(f"file:{MLRUNS}")
mlflow.set_experiment(EXPERIMENT)

with mlflow.start_run(run_name=f"champion-{winner['model']}-full"):
    mlflow.log_param('model', winner['model'])
    mlflow.log_param('family', winner['family'])
    mlflow.log_param('params', json.dumps(winner.get('params')))
    mlflow.log_param('n_train_full', len(X))
    mlflow.log_metric('holdout_rmsle', float(winner['holdout_rmsle']))
    mlflow.log_metric('fit_time_s_full', float(fit_s))
    signature = infer_signature(X, champion.predict(X))
    model_info = mlflow.sklearn.log_model(
        champion, name='pipeline', signature=signature,
        input_example=X.head(2), registered_model_name=REGISTERED_MODEL,
    )

client = MlflowClient()
client.set_registered_model_alias(REGISTERED_MODEL, 'Production', model_info.registered_model_version)
print(f"Modèle enregistré : {REGISTERED_MODEL} v{model_info.registered_model_version} (alias @Production)")

# Vérification : le modèle se recharge depuis le registre et prédit (round-trip)
loaded = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}@Production")
assert np.allclose(loaded.predict(X.head()), champion.predict(X.head())), "round-trip MLflow incohérent"
print("Rechargement models:/inved-house-price@Production vérifié (round-trip OK).")

/home/bogomil/Documents/GitHub/predicting-house-prices-ml/.venv/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/30 13:53:09 INFO mlflow.tracking.fluent: Experiment with name 'inved-house-price' does not exist. Creating a new experiment.


/home/bogomil/Documents/GitHub/predicting-house-prices-ml/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/30 13:53:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization 

Modèle enregistré : inved-house-price v1 (alias @Production)
Rechargement models:/inved-house-price@Production vérifié (round-trip OK).


/home/bogomil/Documents/GitHub/predicting-house-prices-ml/.venv/lib/python3.13/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'inved-house-price'.
Created version '1' of model 'inved-house-price'.


### Servir le champion en API REST

Le modèle sous l'alias `@Production` se sert en **une commande**, comme **processus séparé** (jamais depuis une cellule — cela bloquerait l'exécution headless) :

```bash
mlflow models serve --model-uri "models:/inved-house-price@Production" -p 5000 --env-manager local
```

(`--env-manager local` réutilise le `.venv` du projet ; `--no-conda` est déprécié.) Requête type :

```bash
curl -X POST http://127.0.0.1:5000/invocations \
     -H "Content-Type: application/json" \
     -d '{"dataframe_split": {"columns": ["OverallQual", "GrLivArea", "..."], "data": [[7, 1500, "..."]]}}'
# → {"predictions": [12.05]}   (espace log ; le client applique expm1)
```

En Python, le rechargement tient en une ligne — chemin canonique pour le PoC et les jobs de réentraînement :

```python
import mlflow
champion = mlflow.sklearn.load_model("models:/inved-house-price@Production")
```

Cette architecture (API hébergée, CPU standard) correspond 1:1 à l'engagement du ML Canvas ; en local pour le PoC, transposable telle quelle dans le cloud.

## 6.4 Stratégie de déploiement

> *Architecture cible.* Le registre + le service MLflow sont **implémentés** ci-dessus (§6.3) ; le PoC Streamlit (§6.6) reste à venir. On décrit ici la stratégie de mise en production complète, pas un système déjà exploité en continu.

**Mode d'exécution : temps réel.** L'usage Canvas est une **soumission de formulaire → réponse immédiate** : on sert donc le modèle en **synchrone**, sous le SLA de 5 s (la latence mesurée ~56 ms/requête laisse une marge confortable, NB4 §5.5.3). Un mode **batch** n'est utile que pour un cas secondaire (ré-estimation périodique d'un portefeuille), pas pour le flux principal.

**Chemin d'inférence canonique : MLflow.** Le modèle champion est enregistré dans le **MLflow Model Registry**, puis servi via **`mlflow models serve`** (API REST). Cela colle 1:1 à l'engagement Canvas (« API hébergée dans le cloud, CPU standard ») et rend le monitoring système (couche 2, §7.1) *réel* (latence et taux d'erreur mesurés sur de vraies requêtes HTTP). Le rechargement en une ligne (`mlflow.sklearn.load_model('models:/…@Production')`) couvre les usages hors-ligne (PoC, batch).

**Environnements & CI/CD.** Trois environnements **dev / test / prod**. Le pipeline CI/CD : (1) tests de **schéma** des entrées (colonnes, types), (2) **non-régression RMSLE** sur un holdout figé (refus de promotion si RMSLE > 0,13), (3) build de l'image, (4) promotion en *Staging* puis *Production* dans le Registry.

**Versioning & rollback.** Chaque réentraînement crée une **version** dans le Registry ; les *stages* (Staging/Production/Archived) permettent un **rollback immédiat** (repointer la version précédente) si une dérive est détectée. Les événements de déploiement alimentent la couche 2 du monitoring (§7.1).

## 6.5 Modèle de coût cloud

> Chiffres **estimés** (ordre de grandeur), sauf la latence — seule grandeur réellement mesurée (NB4 §5.5.3). Le but est le *raisonnement*, pas la facture exacte.

| Poste | Ordre de grandeur | Commentaire |
|---|---|---|
| **Entraînement** (compute) | ~nul | Réentraînement mensuel ~13 s sur CPU → quelques centimes/mois. |
| **Service API** (instance always-on) | **poste dominant** | Une instance CPU standard disponible 24/7 pour tenir le SLA < 5 s — c'est le vrai coût récurrent. |
| **Stockage + Registry** | négligeable | Artefacts modèles + logs ; quelques Go. |
| **Monitoring** (logs, dashboards, alertes) | modéré | Rétention 90 j des logs de prédiction (audit équité) + tableaux de bord. |
| **Validation consultant** | réel mais humain | Hors cloud : c'est du temps-homme, pas du compute. |

**Conclusion contre-intuitive.** Le **modèle est quasi gratuit** ; le coût récurrent réel est la **disponibilité de l'API + le monitoring + le workflow de validation humaine**. Le « surcoût » du **Stacking** face à un modèle unique n'est donc **pas le calcul** (latence négligeable face au SLA) mais sa **surface de monitoring** : 4 sous-modèles à surveiller pour la dérive plutôt qu'un seul. C'est l'argument économique du repli vers CatBoost / XGBoost tuné si la maintenabilité prime.

## 6.6 Proof of Concept (bonus)

> **Bonus** — hors périmètre noté, *à venir* (cf. dossier `app/`). Le PoC concrétise l'engagement UX du ML Canvas : formulaire web → prix estimé + intervalle + top-3 facteurs.

**Architecture.** Une application **Streamlit** cliente de l'**API MLflow** servie en §6.3 :

```
Formulaire Streamlit  ──POST /invocations──►  mlflow models serve
   (onglet Prédiction)                         (models:/inved-house-price@Production, :5000)
        ▲                                                 │
        └──────────── prix estimé (expm1) ◄───────────────┘
```

**Lancement (deux processus indépendants, hors noyau du notebook) :**

```bash
# 1. serveur de modèle
mlflow models serve --model-uri "models:/inved-house-price@Production" -p 5000 --env-manager local
# 2. interface (autre terminal)
streamlit run app/app.py
```

**Contenu prévu.** Onglet *Prédiction* (≈10 entrées clés → prix + top-3 SHAP) ; onglet *Monitoring* réalisant le plan en trois couches du §7 (dont la démo « simuler une gentrification »).

*[Captures d'écran à insérer lorsque l'app sera livrée — cf. tâche PoC.]*

## Transition — Phase 6 → Phase 7 (surveillance et maintenance)

Le champion est réentraîné, soumis, enregistré dans le registre MLflow et servi en API ; la stratégie de déploiement, le coût et le PoC sont posés. Mais déployer n'est pas la fin : un modèle de prix immobilier vieillit dès que le marché bouge. **CRISP-ML(Q)** ajoute donc une **septième phase** — *surveillance et maintenance* — développée dans le NB6 : savoir en continu si le modèle reste bon (couche mathématique), si l'API tient ses promesses (couche système) et s'il crée la valeur métier promise (couche métier).